<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex02-pytorch-and-autograd/Ex02_00_environment_check.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference text — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_2 · Notebook 00 — environment check

**Read-only. There is nothing to complete in this notebook.** Run it top to
bottom. If every cell runs, your PyTorch installation is ready.

Ex_1 checked NumPy and matplotlib. This one checks the library the rest of the
course is written in, and it ends with a six-line demonstration of the single
idea that Part 2 is built on — differentiating a function with respect to its
input — so that you have seen it work before notebook 02 asks you to reason
about it.

## What PyTorch is, in one paragraph

It is an array library, like NumPy, with two additions. First, arrays — called
**tensors** — can live on a GPU, and an operation on them runs there. Second,
and far more important for this course, a tensor can be asked to **remember
what happened to it**: every operation involving it is recorded in a graph, and
that graph can be walked backwards to produce exact derivatives of any result
with respect to any input.

The second feature is usually presented as the thing that trains neural
networks. It is, but that is a special case. It is a general differentiation
engine, and once you see it that way you can use it to evaluate the residual of
a partial differential equation — which is what the whole of L7 to L12 does.

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_2_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex02-pytorch-and-autograd/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup -------------------------------------------------------------
# Needs Ex_2_core.py alongside this notebook.
import os
for f in ("Ex_2_core.py",):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from Ex_2_core import *                             # noqa: F401,F403
import numpy as np
import torch
import matplotlib.pyplot as plt

set_seed(88)
print("setup complete | device:", DEVICE, "| dtype:", DTYPE)

**What you should see.** `setup complete | device: cpu | dtype: torch.float32`.

If you get a `ModuleNotFoundError` for `torch`, install it — the CPU build is
the right one:

```
pip install torch --index-url https://download.pytorch.org/whl/cpu
```

On Colab it is preinstalled and this cannot happen.

## 1 · Versions and device

Three things are reported below.

**The torch version.** Anything from 1.12 onwards runs these notebooks. The
autograd interface used here has been stable for years.

**Whether CUDA is available.** Almost certainly `False`, and that is fine —
`DEVICE` in `Ex_2_core.py` is fixed to the CPU because every problem in this
exercise is tiny. A network with two thousand parameters trained on sixty data
points spends more time moving data to a GPU than computing on it. Part 2 uses
the same variable name for a device that may well be a GPU, and notebook 01
shows how that line is normally written.

**The default dtype.** PyTorch defaults to `float32`, NumPy to `float64`. This
is the single most common source of type errors when the two libraries meet,
and this course states the type explicitly everywhere to avoid it.

In [ ]:
banner("PyTorch")
print("  torch version     ", torch.__version__)
print("  CUDA available    ", torch.cuda.is_available())
print("  DEVICE (this course)", DEVICE)
print("  DTYPE  (this course)", DTYPE)
print("  torch default dtype ", torch.get_default_dtype())
print("  numpy default       ", np.zeros(1).dtype)
print()
print("  threads torch will use:", torch.get_num_threads())

**What you should see.** A version number, `CUDA available False` on most
machines, `cpu`, and `torch.float32` against NumPy's `float64`.

The thread count is reported because it explains why a timing measured on
Colab differs from one measured on your laptop. It has no effect on any result.

## 2 · Tensors behave like arrays

Everything you learned in Ex_1 notebook 02 about shape, dtype, indexing and
broadcasting transfers directly. The rule is the same rule: align shapes from
the right, every pair equal or containing a 1.

The names differ in a few places — `dim` instead of `axis`, `.reshape` and
`.view` where NumPy has only `.reshape` — and notebook 01 goes through them.
For now, just confirm that arithmetic works and shapes do what you expect.

In [ ]:
a = torch.linspace(0.0, 1.0, 5)
b = torch.ones(3, 1)

print("a          ", a, "shape", tuple(a.shape))
print("b shape    ", tuple(b.shape))
print("a + b shape", tuple((a + b).shape), " <- broadcasting, exactly as in NumPy")
print()
print("a.dtype    ", a.dtype)
print("a.mean()   ", a.mean().item(), " .item() pulls a one-element tensor out as a float")

**What you should see.** Five values from 0 to 1, `b` with shape `(3, 1)`, and
their sum with shape `(3, 5)`. If that last shape surprises you, go back to
Ex_1 notebook 02 §4 before continuing — everything from here on depends on it.

## 3 · Seeds

`set_seed(88)` seeds three generators: Python's `random`, NumPy's, and
PyTorch's. The third is the one that decides the initial weights of a network,
so without it no training run is comparable with any other.

Run the cell twice; the numbers do not change.

In [ ]:
set_seed(88)
first = torch.randn(3)
set_seed(88)
again = torch.randn(3)

print("first :", first)
print("again :", again)
print("identical:", bool(torch.equal(first, again)))

net = mlp(n_in=1, n_out=1, n_hidden=16, n_layers=2)
print()
print("a small network:", count_parameters(net), "parameters")

**What you should see.** Two identical rows of three numbers, `identical: True`,
and a network with **321** parameters.

That count is worth a moment's arithmetic, because knowing where parameters
live is the beginning of knowing why a model is the wrong size. Two hidden
layers of 16 units, one input, one output:

| layer | weights | biases |
|---|---|---|
| 1 → 16 | 16 | 16 |
| 16 → 16 | 256 | 16 |
| 16 → 1 | 16 | 1 |

32 + 272 + 17 = 321. Note where the parameters actually are: the middle layer
holds five sixths of them, and widening the network grows that term
quadratically.

## 4 · The smoke test that matters

Here is the whole of Part 2, in six lines.

Take the function u(x) = x³. Mark the input as something to be tracked. Compute
u. Then ask autograd for du/dx.

The answer must be 3x², which at x = 2 is 12. Not approximately 12 — exactly
12, up to floating-point rounding, because autograd does not approximate the
derivative. It applies the chain rule to the operations that were actually
performed, so the result is the derivative of the function you wrote, not of a
nearby one.

Compare that with a finite difference, which is what you would otherwise do:
evaluate the function twice, a distance h apart, and divide. That answer
depends on h, and getting it to five digits requires choosing h well.

In [ ]:
x = torch.tensor([2.0], requires_grad=True)      # mark the INPUT
u = x ** 3                                       # the function
u_x = torch.autograd.grad(u, x)[0]               # du/dx

print("  x        ", x.item())
print("  u = x^3  ", u.item())
print("  du/dx    ", u_x.item(), "   (autograd)")
print("  3 x^2    ", 3 * 2.0 ** 2, "   (by hand)")
print()
h = 1e-3
fd = ((2.0 + h) ** 3 - (2.0 - h) ** 3) / (2 * h)
print(f"  finite difference, h = {h}: {fd:.9f}   (close, and not exact)")

**What you should see.**

```
  du/dx     12.0    (autograd)
  3 x^2     12.0    (by hand)
  finite difference, h = 0.001: 12.000001000   (close, and not exact)
```

The finite difference is wrong in the seventh digit, and it would be wrong in a
different digit for a different h. Autograd has no h.

That is the entire idea. Notebook 02 takes it apart properly: what
`grad_outputs` is for, why `create_graph=True` is needed for a second
derivative, and what happens when the function is a neural network rather than
a cube.

## 5 · matplotlib, and the convention for loss curves

One figure, to confirm plotting works and to state a convention.

**Every loss curve in this course is plotted on a logarithmic y-axis.** A loss
falls by orders of magnitude; on a linear axis everything after the first
decade is flattened onto the baseline, and you will conclude that training
stalled when it in fact improved by a factor of a thousand. `plot_loss` in the
core module sets the log scale for you, and it is not optional.

In [ ]:
fake_history = 1.0 * np.exp(-np.arange(500) / 60.0) + 1e-4

fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.4))
axes[0].plot(fake_history)
engineering_axes(axes[0], "epoch [-]", "loss [-]", title="linear y-axis: looks stalled")
plot_loss(fake_history, ax=axes[1], label="loss", title="log y-axis: the truth")
fig.tight_layout()
plt.show()

**What you should see.** Two panels of the same data. On the left, a curve that
drops and then appears to flatline at zero. On the right, a straight line
falling by four decades and then levelling off at the floor of 1e-4 — where you
can see both the exponential rate and exactly when it stopped improving.

Only the right-hand panel lets you answer "is it still learning?", which is the
question you will actually be asking.

## 6 · You are ready

If every cell above ran, you have a working PyTorch, reproducible seeds, and
you have seen autograd produce an exact derivative.

## Where to go next

| | |
|---|---|
| `Ex02_01_tensors.ipynb` | tensors, dtype and device, shapes, NumPy interop, and what `requires_grad` actually does |
| `Ex02_02_autograd_by_hand.ipynb` | **the most important notebook in Part 1** |
| `Ex02_03_training_loop.ipynb` | forward, loss, zero_grad, backward, step — fitting a curve |

Notebook 02 is not optional and it is not a warm-up. Everything you do from
week seven to the end of the course is the two functions it teaches, applied to
a different equation.